# MNIST Binary Classification: Digits 1 vs 7

**Student:** Khush Bhavin Maru

**Course:** Machine Learning Algorithms


This notebook covers binary classification on a subset of the MNIST
dataset containing only digits 1 and 7. The training set has 100
labeled images and the test set has 1000 labeled images.

### Environment check

Run this cell first to confirm all required packages are installed
and print their versions. If anything is missing, install it with:
pip install -r requirements.txt

In [ ]:
# environment check, run this before anything else
import importlib
import sys

required = {
    'numpy'       : 'numpy',
    'matplotlib'  : 'matplotlib',
    'seaborn'     : 'seaborn',
    'sklearn'     : 'scikit-learn',
    'notebook'    : 'notebook',
}

print(f'Python version : {sys.version.split()[0]}')
print()

all_good = True
for module, package in required.items():
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, '__version__', 'version not found')
        print(f'  {package:<20} {version}  OK')
    except ImportError:
        print(f'  {package:<20} NOT INSTALLED  run: pip install {package}')
        all_good = False

print()
if all_good:
    print('All packages found. Good to go.')
else:
    print('Some packages are missing. Install them before running further.')

## Task 1: Data Exploration and Analysis

In this section we load the dataset, inspect its structure, compute
descriptive statistics, and visualise key characteristics of both
classes before any modelling is done.

In [ ]:
import sys
import warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_data
from preprocessor import flatten, get_scaler, apply_scaler, binarise_labels
from stats import class_counts, class_balance_ratio, pixel_stats_per_class, print_summary
from visualiser import plot_sample_images, plot_pixel_stats, plot_class_distribution

np.random.seed(42)

In [ ]:
# load the dataset from the data folder
X_train_raw, y_train, X_test_raw, y_test = load_data(
    '../data/mnist_subsets.npz'
)

print('Dataset loaded successfully')
print()
print(f'X_train shape : {X_train_raw.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'X_test shape  : {X_test_raw.shape}')
print(f'y_test shape  : {y_test.shape}')
print()
print(f'Pixel dtype   : {X_train_raw.dtype}')
print(f'Pixel range   : {X_train_raw.min()} to {X_train_raw.max()}')
print(f'Unique labels : {np.unique(y_train).tolist()}')

In [ ]:
# check class counts and balance in both splits
print('Training set')
print(f'  Class counts  : {class_counts(y_train)}')
print(f'  Balance ratio : {class_balance_ratio(y_train)}')
print()
print('Test set')
print(f'  Class counts  : {class_counts(y_test)}')
print(f'  Balance ratio : {class_balance_ratio(y_test)}')
print()
print('A balance ratio of 1.0 means both classes have equal samples.')
print('The training set is perfectly balanced so no class weighting is needed.')

In [ ]:
# compute pixel-level stats for each digit in the training set
stats = pixel_stats_per_class(X_train_raw, y_train)

for digit, s in stats.items():
    print(f'Digit {digit}  (n={s["n_samples"]} samples)')
    print(f'  pixel mean     : {s["pixel_mean"]}')
    print(f'  pixel std      : {s["pixel_std"]}')
    print(f'  pixel min/max  : {s["pixel_min"]} / {s["pixel_max"]}')
    print(f'  non-zero pixels: {s["nonzero_pct"]}%')
    print()

### Observations from pixel statistics

- Both digits use the full pixel range (0 to 255), so no clipping is needed.
- Digit 7 has a noticeably higher pixel mean (29.53 vs 20.18) and nearly
  1.5x more non-zero pixels (16.76% vs 11.27%). This reflects the structural
  difference between the two digits: digit 7 has a horizontal top stroke and
  a diagonal, covering more pixel area than the thin vertical stroke of digit 1.
- The higher pixel std for digit 7 (74.88 vs 63.75) reflects greater variation
  in how people write it, including crossed vs uncrossed variants and differences
  in stroke width and slant.
- These differences confirm strong linear separability in pixel space, which
  directly supports the choice of Logistic Regression in Task 2.
- Pixel values span 0 to 255 so standardisation (zero mean, unit variance)
  will be applied before training to stop high-magnitude pixels from
  disproportionately influencing gradient-based and margin-based models.

### Class imbalance note

The training set is perfectly balanced: 50 digit 1 and 50 digit 7 samples.
However the test set is imbalanced: 100 digit 1 and 900 digit 7 samples
(balance ratio 0.11). This is important for evaluation. A model that always
predicts digit 7 would achieve 90% accuracy on the test set while completely
failing on digit 1. This is exactly why macro F1 is used as the primary
evaluation metric throughout Tasks 2 and 3 alongside accuracy. Macro F1
weights both classes equally regardless of their frequency in the test set,
so it catches this kind of bias that accuracy alone would miss.

### Visualisation 1: Sample training images

Eight randomly selected examples from each class. This gives a quick
sanity check that the data loaded correctly and shows the visual
difference between the two digits.

In [ ]:
# plot 8 examples per class from the training set
# saved to outputs folder for the report
plot_sample_images(
    X_train_raw,
    y_train,
    n_per_class=8,
    save_path='../outputs/viz1_sample_images.png'
)

### Visualisation 2: Per-class pixel statistics

Three plots per digit row: the average image across all samples,
the pixel standard deviation map (shows where pixels vary most
across samples), and a histogram of non-zero pixel intensities.

In [ ]:
# mean image, std map, and intensity histogram for each digit
plot_pixel_stats(
    X_train_raw,
    y_train,
    save_path='../outputs/viz2_pixel_stats.png'
)

### Visualisation 3: Class distribution

Bar chart comparing how many samples of each digit exist in the
training and test splits.

In [ ]:
# class distribution across train and test
plot_class_distribution(
    y_train,
    y_test,
    save_path='../outputs/viz3_class_distribution.png'
)

### Preprocessing

Before training, two preprocessing steps are applied:

1. **Flatten**: each 28x28 image is reshaped into a 784-length
   vector so standard classifiers can consume it directly.

2. **Standardise**: pixel values are scaled to zero mean and unit
   variance using StandardScaler. The scaler is fitted on the training
   set only and then applied to the test set. This stops
   high-intensity pixels from disproportionately influencing
   gradient-based and margin-based models.

Labels are also converted from the original values (1 and 7) to
binary values (0 and 1) where digit 1 maps to 0 and digit 7 maps
to 1.

### Task 1 summary

The key findings from data exploration that directly inform the
modelling decisions in Tasks 2 and 3:

- The training set is perfectly balanced (50 samples per class),
  so no class weighting is needed during training.
- The test set is imbalanced (100 digit 1, 900 digit 7), which is
  why macro F1 is used as the primary evaluation metric. Accuracy
  alone would be misleading on this test distribution.
- Digit 7 has 48% more non-zero pixels (16.76% vs 11.27%) and a
  higher pixel mean (29.53 vs 20.18), confirming the two classes
  are visually and statistically distinct.
- PCA projection shows clean class separation in just 2 dimensions,
  directly supporting the choice of a linear classifier in Task 2.
- All preprocessing was applied correctly: StandardScaler fitted on
  training data only, never on test data, preventing data leakage.

In [ ]:
# flatten images and standardise pixel values
X_train_flat = flatten(X_train_raw)
X_test_flat  = flatten(X_test_raw)

scaler  = get_scaler(X_train_flat)
X_train = apply_scaler(scaler, X_train_flat)
X_test  = apply_scaler(scaler, X_test_flat)

# convert labels to binary
y_train_bin = binarise_labels(y_train)
y_test_bin  = binarise_labels(y_test)

print(f'X_train: {X_train.shape}  mean={X_train.mean():.4f}  std={X_train.std():.4f}')
print(f'X_test : {X_test.shape}')
print(f'y_train classes: {np.unique(y_train_bin).tolist()}')
print(f'y_test  classes: {np.unique(y_test_bin).tolist()}')
print()
print('Preprocessing done. Scaler fitted on training data only.')

### Visualisation 4: PCA projection of training data

Principal Component Analysis (PCA) reduces the 784-dimensional
pixel vectors down to 2 dimensions so we can plot both classes
on a single scatter plot. This gives a visual sense of how
separable the two digit classes are before any classifier is
trained.

If the two classes form distinct clusters in 2D PCA space it
suggests a linear classifier will handle this task well.

In [ ]:
from sklearn.decomposition import PCA

# reduce training data to 2 components for visualisation only
# PCA is fitted on training data only, consistent with the scaler
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train)

# plot the two classes in PCA space
fig, ax = plt.subplots(figsize=(8, 6))
for label_bin, digit, colour in zip([0, 1], [1, 7], ['#4C72B0', '#DD8452']):
    mask = y_train_bin == label_bin
    ax.scatter(
        X_train_pca[mask, 0],
        X_train_pca[mask, 1],
        label=f'Digit {digit}',
        color=colour,
        alpha=0.85,
        edgecolors='white',
        linewidths=0.5,
        s=80
    )

ax.set_title(
    'PCA projection of training data (784 dims reduced to 2)',
    fontsize=12
)
ax.set_xlabel(
    f'PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% variance)'
)
ax.set_ylabel(
    f'PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% variance)'
)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/viz4_pca.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print(f'PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance')
print(f'Combined    {sum(pca.explained_variance_ratio_)*100:.1f}% of variance')

## Task 2: Build a Classification System

Here we train a Logistic Regression classifier on the training set,
justify the model and metric choices, and report performance on the
test set.

### Model choice: Logistic Regression

Logistic Regression was chosen for the following reasons:

- The training set is small (100 samples). A simple linear model is
  less likely to overfit compared to more complex models on this
  amount of data.
- Digits 1 and 7 have clearly different shapes. Digit 1 is a thin
  vertical stroke while digit 7 has a horizontal bar and a diagonal.
  This kind of structural difference tends to be linearly separable
  in pixel space after standardisation.
- Logistic Regression has a well understood regularisation parameter
  C that gives direct control over model complexity.
- It produces calibrated class probabilities which makes the output
  interpretable.

### Evaluation metric: Accuracy and Macro F1

- **Macro F1** is the primary evaluation metric for this task.
  The test set has 100 digit 1 samples and 900 digit 7 samples.
  A model that predicts digit 7 for everything would score 90%
  accuracy while completely failing on digit 1. Macro F1 averages
  the F1 score per class with equal weight, so it catches exactly
  this kind of imbalance-driven inflation. It is the correct metric
  when both classes matter equally regardless of their frequency.
- **Accuracy** is reported alongside macro F1 as a secondary metric
  because it is easy to interpret and gives an overall picture of
  how many samples were classified correctly. The two metrics together
  give a complete view: accuracy for overall correctness, macro F1
  for per-class fairness.

In [ ]:
from logistic_model import build_lr, train, evaluate

# train logistic regression with default C=1.0 on the training set
lr_model = build_lr(C=1.0)
lr_model = train(lr_model, X_train, y_train_bin)

print('Model trained on training set only.')
print(f'Number of training samples: {X_train.shape[0]}')
print(f'Number of features: {X_train.shape[1]}')

In [ ]:
# evaluate on the test set
y_pred_lr, acc_lr, f1_lr = evaluate(lr_model, X_test, y_test_bin, split_name='Test')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# confusion matrix for logistic regression
cm = confusion_matrix(y_test_bin, y_pred_lr)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted 1', 'Predicted 7'],
    yticklabels=['True 1', 'True 7'],
    ax=ax
)
ax.set_title('Confusion matrix  Logistic Regression (test set)', fontsize=11)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()
plt.savefig('../outputs/viz4_lr_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

### Weight visualisation

One advantage of Logistic Regression over black-box models is that
the learned weights are directly interpretable. The weight vector
has the same length as the input (784 values) and can be reshaped
back to 28x28 to see which pixels the model relies on most.

Red pixels push the prediction toward digit 7. Blue pixels push
toward digit 1. Grey pixels near zero had little influence on the
decision boundary.

In [ ]:
from visualiser import plot_lr_weights

# visualise the learned weights of the logistic regression model
plot_lr_weights(
    lr_model,
    save_path='../outputs/viz_lr_weights.png'
)

## Task 3: Comparison of Two Classifiers and Hyperparameter Tuning

We introduce a second classifier (Kernel SVM with RBF kernel) and
run cross-validation on the training set to tune hyperparameters for
both models. The test set is not touched until final evaluation.

### Cross-validation and model selection strategy

5-fold Stratified Cross-Validation is used on the 100-sample training
set for all hyperparameter tuning. Here is exactly how it works:

- The training set is split into 5 folds of 20 samples each.
- In each round, 4 folds (80 samples) are used for fitting and 1 fold
  (20 samples) is held out for validation.
- This repeats 5 times so every training sample is used for validation
  exactly once, giving a reliable estimate of generalisation performance.
- Stratified means each fold preserves the 50/50 class ratio of the full
  training set. On only 100 samples, a non-stratified random split could
  accidentally produce unbalanced folds that give misleading CV scores.

The test set is not used at any point during tuning or model selection.
It is only used for final evaluation after the best hyperparameters have
already been chosen from CV results alone.

Macro F1 is used as the CV selection metric rather than accuracy. This is
a deliberate choice: the test set has 9x more digit 7 than digit 1, so
a model that focuses on digit 7 at the expense of digit 1 could score
well on accuracy while performing poorly on one class. Macro F1 weights
both classes equally and prevents this kind of biased selection.

In [ ]:
from tuner import tune_lr

# tune logistic regression C value using 5-fold stratified CV
# C controls regularisation: small C = strong regularisation
print('Tuning Logistic Regression over C values...')
print()
lr_C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
lr_results, best_C_lr = tune_lr(X_train, y_train_bin, C_values=lr_C_values)

In [ ]:
# plot CV F1 score vs C value for logistic regression
lr_C_list  = [r[0] for r in lr_results]
lr_f1_list = [r[1] for r in lr_results]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lr_C_list, lr_f1_list, marker='o', color='steelblue', linewidth=2)
ax.axvline(best_C_lr, color='crimson', linestyle='--',
           label=f'Best C = {best_C_lr}')
ax.set_title('Logistic Regression  CV macro F1 vs C', fontsize=12)
ax.set_xlabel('C (log scale)')
ax.set_ylabel('Mean CV macro F1')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/viz5_lr_cv.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Best C selected: {best_C_lr}')

### SVM hyperparameter search

Two parameters are tuned for the RBF SVM:

- **C**: controls the trade-off between a wider margin and correctly
  classifying training points. A small C allows more misclassifications
  in exchange for a smoother boundary. A large C tries to classify
  everything correctly but risks overfitting.
- **gamma**: controls the reach of each training sample. A small gamma
  means each sample has a broad influence on the boundary. A large
  gamma means very local influence, which can lead to overfitting on
  small datasets.

A 5x5 grid search is run across both parameters using the same
5-fold stratified CV setup as the LR search above.

In [ ]:
from tuner import tune_svm

# grid search over C and gamma for RBF SVM
print('Tuning SVM over C x gamma grid...')
print()
svm_C_values     = [0.01, 0.1, 1, 10, 100]
svm_gamma_values = [0.0001, 0.001, 0.01, 0.1, 1]

svm_grid, best_C_svm, best_gamma_svm = tune_svm(
    X_train, y_train_bin,
    C_values=svm_C_values,
    gamma_values=svm_gamma_values
)

In [ ]:
# heatmap of CV macro F1 across the C x gamma grid
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(
    svm_grid, cmap='YlOrRd', aspect='auto',
    vmin=svm_grid.min(), vmax=svm_grid.max()
)
ax.set_xticks(range(len(svm_gamma_values)))
ax.set_xticklabels(svm_gamma_values, rotation=45, ha='right')
ax.set_yticks(range(len(svm_C_values)))
ax.set_yticklabels(svm_C_values)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.set_title('SVM RBF  CV macro F1 across C x gamma grid', fontsize=12)

# write the score in each cell
for i in range(len(svm_C_values)):
    for j in range(len(svm_gamma_values)):
        ax.text(j, i, f'{svm_grid[i, j]:.3f}',
                ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, label='Mean CV macro F1')
plt.tight_layout()
plt.savefig('../outputs/viz6_svm_cv_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Best C: {best_C_svm}  |  Best gamma: {best_gamma_svm}')

### Final chosen hyperparameters

Based on 5-fold stratified cross-validation on the training set,
the following hyperparameters were selected for each model:

| Model | Parameter | Value | CV Macro F1 |
|-------|-----------|-------|-------------|
| Logistic Regression | C | 0.1 | 1.0000 |
| Kernel SVM RBF | C | 100 | 0.9900 |
| Kernel SVM RBF | gamma | 0.0001 | 0.9900 |

For Logistic Regression, C=0.1 was chosen because it achieves the
maximum CV score with the strongest regularisation among all tied
values. On a 100-sample dataset, preferring stronger regularisation
reduces overfitting risk while maintaining full cross-validation
performance.

For the SVM, C=100 with gamma=0.0001 was the best combination in
the 5x5 grid. The small gamma (broad influence per sample) combined
with high C (tight margin) is a common optimal configuration when
classes are well separated but the training set is small. Very large
gamma values all scored below 0.47, confirming that overly local
decision boundaries overfit on 100 training samples.

### Final model training and comparison

Both models are retrained on the full 100-sample training set using
the chosen hyperparameters above and evaluated on the test set for
the first time.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
from logistic_model import build_lr, train, evaluate as lr_evaluate
from svm_model import build_svm, train as svm_train, evaluate as svm_evaluate

# retrain logistic regression with best C from CV
lr_final  = build_lr(C=best_C_lr)
lr_final  = train(lr_final, X_train, y_train_bin)

# retrain SVM with best C and gamma from CV
svm_final = build_svm(C=best_C_svm, gamma=best_gamma_svm)
svm_final = svm_train(svm_final, X_train, y_train_bin)

print(f'LR  trained with C={best_C_lr}')
print(f'SVM trained with C={best_C_svm}, gamma={best_gamma_svm}')

In [ ]:
# evaluate both models on the test set
print('=== Logistic Regression ===')
y_pred_lr_final, acc_lr_f, f1_lr_f = lr_evaluate(
    lr_final, X_test, y_test_bin, split_name='Test'
)

print('=== Kernel SVM RBF ===')
y_pred_svm_final, acc_svm_f, f1_svm_f = svm_evaluate(
    svm_final, X_test, y_test_bin, split_name='Test'
)

In [ ]:
# side by side confusion matrices for both final models
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, preds, title in zip(
    axes,
    [y_pred_lr_final, y_pred_svm_final],
    [f'Logistic Regression  (C={best_C_lr})',
     f'Kernel SVM RBF  (C={best_C_svm}, gamma={best_gamma_svm})']
):
    cm = confusion_matrix(y_test_bin, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Predicted 1', 'Predicted 7'],
        yticklabels=['True 1', 'True 7'],
        ax=ax
    )
    acc = accuracy_score(y_test_bin, preds)
    ax.set_title(f'{title}\nAccuracy = {acc:.4f}', fontsize=10)
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')

plt.suptitle('Final model comparison on test set', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/viz7_final_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

### Per-class accuracy breakdown

Overall accuracy can hide imbalances in how well a model performs
on each individual class. This breakdown shows the per-class
accuracy for both models so we can see if either model struggles
more with one digit than the other.

In [ ]:
# per-class accuracy for both final models
models_info = [
    ('Logistic Regression', y_pred_lr_final),
    ('Kernel SVM RBF',      y_pred_svm_final),
]

print(f'{"Model":<28} {"Digit 1 acc":>12} {"Digit 7 acc":>12} {"Overall acc":>12}')
print('-' * 68)

for name, preds in models_info:
    mask_1 = y_test_bin == 0
    mask_7 = y_test_bin == 1

    acc_1   = accuracy_score(y_test_bin[mask_1], preds[mask_1])
    acc_7   = accuracy_score(y_test_bin[mask_7], preds[mask_7])
    acc_all = accuracy_score(y_test_bin, preds)

    print(f'{name:<28} {acc_1:>12.4f} {acc_7:>12.4f} {acc_all:>12.4f}')

print()
print('Digit 1 acc = how many actual digit 1s were correctly predicted')
print('Digit 7 acc = how many actual digit 7s were correctly predicted')

### What the per-class breakdown shows

- A model with high overall accuracy but low per-class accuracy on
  one digit is biased toward the majority class. That is not the
  case here since both classes are balanced.
- If digit 1 accuracy is lower it usually means the model is
  confusing some digit 1s for digit 7s, likely on samples where
  the handwriting is slightly slanted or ambiguous.
- Comparing both models on this breakdown tells us which one is
  more consistent across classes, not just better on average.

### Model comparison results

| Model | Best params | Test accuracy | Test macro F1 |
|-------|-------------|---------------|---------------|
| Logistic Regression | C=0.1 | 0.9460 | 0.8740 |
| Kernel SVM RBF | C=100, gamma=0.0001 | 0.9480 | 0.8723 |

| Model | Digit 1 accuracy | Digit 7 accuracy | Overall accuracy |
|-------|-----------------|-----------------|-----------------|
| Logistic Regression | 0.9500 | 0.9456 | 0.9460 |
| Kernel SVM RBF | 0.8900 | 0.9544 | 0.9480 |

### Discussion

Both models achieve nearly identical overall accuracy (94.6% vs
94.8%), which is itself the key finding: it confirms the decision
boundary between these two digit classes is approximately linear
in standardised pixel space, consistent with what the PCA and
pixel statistics showed in Task 1. The non-linear RBF boundary
that SVM can learn provides no meaningful advantage here.

The per-class breakdown reveals something the overall accuracy
hides. Logistic Regression correctly classifies 95% of digit 1s
while the SVM only gets 89%. The SVM compensates with slightly
better digit 7 accuracy (95.4% vs 94.6%), which pushes its
overall accuracy marginally higher. But for a task where both
classes matter equally, Logistic Regression is actually the more
consistent model.

From a practical standpoint:

- Logistic Regression trains faster, is fully interpretable through
  its weight vector, and classifies digit 1 more reliably (95% vs 89%).
- The SVM's 0.2% overall accuracy gain does not justify its higher
  computational cost, lower interpretability, and worse digit 1
  performance.
- Both models confirm the dataset is linearly separable. A non-linear
  approach is not needed for this particular binary task.

## Task 4: Clustering Analysis of the Test Set

We extract digit-7 samples from the test set and apply K-means
clustering. We first cluster into 100 groups, then find the optimal
number of clusters using the elbow method and silhouette score.

### Setup: extract digit 7 from the test set

The test set is used here in an unsupervised way. Labels are only
used to filter out the digit 7 samples before clustering. No label
information is passed into K-means at any point.

The same scaler fitted on the training set in Task 1 is reused here
to standardise the digit 7 test samples before clustering. This is
correct practice since the scaler was never fitted on test data.

In [ ]:
from clusterer import run_kmeans, compute_elbow, compute_silhouette
from clusterer import get_optimal_k, get_cluster_centers_pixels

# extract digit 7 samples from the test set using labels only for filtering
idx_7    = np.where(y_test == 7)[0]
X_test_7_flat = X_test_flat[idx_7]
X_test_7      = apply_scaler(scaler, X_test_7_flat)

print(f'Digit 7 test samples available: {X_test_7.shape[0]}')
print(f'Feature dimensions: {X_test_7.shape[1]}')

### Step 1: K-means with k=100

As specified in the task, we first cluster the digit 7 test samples
into exactly 100 clusters using K-means. Inertia (total within-cluster
sum of squared distances) is reported to confirm the model ran correctly.

In [ ]:
# run k-means with k=100 on the digit 7 test samples
km100, labels100 = run_kmeans(X_test_7, n_clusters=100)

cluster_sizes_100 = np.bincount(labels100)

print(f'K-means with k=100 complete')
print(f'Inertia          : {km100.inertia_:.2f}')
print(f'Largest cluster  : {cluster_sizes_100.max()} samples')
print(f'Smallest cluster : {cluster_sizes_100.min()} samples')
print(f'Mean cluster size: {cluster_sizes_100.mean():.1f} samples')

### Step 2: Finding the optimal number of clusters

Two methods are used together to identify the optimal k:

- **Elbow method**: plots inertia against k. Inertia drops sharply
  at first then flattens out. The point where the curve bends like
  an elbow is a reasonable choice for k since adding more clusters
  beyond that point gives diminishing returns.
- **Silhouette score**: measures how well each sample fits its own
  cluster compared to the nearest neighbouring cluster. Scores range
  from -1 to 1 where higher means better defined clusters. The k
  with the highest silhouette score is selected as optimal.

Both methods are run across k=2 to k=40 on the digit 7 test samples.

In [ ]:
# compute elbow and silhouette scores across k=2 to k=40
k_range = range(2, 41)

print('Computing elbow curve...')
inertias = compute_elbow(X_test_7, k_range)

print('Computing silhouette scores...')
sil_scores = compute_silhouette(X_test_7, k_range, sample_size=300)

optimal_k, optimal_sil = get_optimal_k(k_range, sil_scores)

print()
print(f'Optimal k by silhouette score: {optimal_k}')
print(f'Silhouette score at optimal k: {optimal_sil}')

In [ ]:
# plot elbow curve and silhouette scores side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# elbow curve
axes[0].plot(list(k_range), inertias, marker='o', markersize=4,
             color='steelblue', linewidth=2)
axes[0].set_title('Elbow method  inertia vs k', fontsize=12)
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].grid(alpha=0.3)

# silhouette score
axes[1].plot(list(k_range), sil_scores, marker='o', markersize=4,
             color='darkorange', linewidth=2)
axes[1].axvline(optimal_k, color='crimson', linestyle='--',
                label=f'Optimal k = {optimal_k}')
axes[1].set_title('Silhouette score vs k', fontsize=12)
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Mean silhouette score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(
    'Optimal cluster count for digit 7 test samples',
    fontsize=12
)
plt.tight_layout()
plt.savefig('../outputs/viz8_elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Selected k={optimal_k} based on highest silhouette score of {optimal_sil}')

### Step 3: Fit K-means with optimal k

The optimal k identified from the silhouette analysis is used to
fit a final K-means model on the digit 7 test samples. Cluster
centres are then inverse-transformed back to pixel space so they
can be visualised as 28x28 images.

In [ ]:
# fit final k-means with the optimal k
km_opt, labels_opt = run_kmeans(X_test_7, n_clusters=optimal_k)

cluster_sizes = np.bincount(labels_opt)

# inverse transform cluster centres back to pixel space for display
centers_px = get_cluster_centers_pixels(km_opt, scaler)

print(f'Final K-means fitted with k={optimal_k}')
print(f'Cluster sizes: min={cluster_sizes.min()}  '
      f'max={cluster_sizes.max()}  '
      f'mean={cluster_sizes.mean():.1f}')

### Visualisation: cluster centres

Each panel below shows one cluster centre reshaped back to 28x28
pixels. The title shows the cluster index and how many digit 7
samples belong to that cluster. Clusters are sorted by size so the
most populated ones appear first.

In [ ]:
# plot all cluster centres sorted by cluster size (largest first)
n_centers = len(centers_px)
order     = np.argsort(-cluster_sizes)

cols = min(10, n_centers)
rows = int(np.ceil(n_centers / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9))
axes = np.array(axes).flatten()

for plot_i, cluster_i in enumerate(order):
    ax = axes[plot_i]
    ax.imshow(centers_px[cluster_i], cmap='gray_r', interpolation='nearest')
    ax.set_title(
        f'C{cluster_i}\nn={cluster_sizes[cluster_i]}',
        fontsize=7
    )
    ax.axis('off')

for ax in axes[n_centers:]:
    ax.set_visible(False)

plt.suptitle(
    f'K-means cluster centres  k={optimal_k}  digit 7 test samples\n'
    f'sorted by cluster size, largest first',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../outputs/viz9_cluster_centres.png', dpi=120, bbox_inches='tight')
plt.show()

### Visualisation: representative samples per cluster

Instead of just the average centre image, this plot shows the actual
digit 7 sample from the test set that sits closest to each cluster
centroid. This gives a more concrete view of what each cluster
actually contains.

In [ ]:
# for each cluster find the real sample closest to the centroid
n_show = min(optimal_k, 20)
order2 = np.argsort(-cluster_sizes)[:n_show]

cols2 = min(10, n_show)
rows2 = int(np.ceil(n_show / cols2))

fig, axes = plt.subplots(rows2, cols2, figsize=(cols2 * 1.6, rows2 * 1.9))
axes = np.array(axes).flatten()

for plot_i, cluster_i in enumerate(order2):
    # find index of sample in this cluster closest to the centroid
    mask     = labels_opt == cluster_i
    members  = X_test_7[mask]
    centroid = km_opt.cluster_centers_[cluster_i]
    dists    = np.linalg.norm(members - centroid, axis=1)
    rep_idx  = np.where(mask)[0][np.argmin(dists)]

    ax = axes[plot_i]
    ax.imshow(
        X_test_7_flat[rep_idx].reshape(28, 28),
        cmap='gray_r', interpolation='nearest'
    )
    ax.set_title(
        f'C{cluster_i}\nn={cluster_sizes[cluster_i]}',
        fontsize=7
    )
    ax.axis('off')

for ax in axes[n_show:]:
    ax.set_visible(False)

plt.suptitle(
    f'Most representative digit 7 per cluster  (top {n_show} by size)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../outputs/viz10_representative_samples.png', dpi=120, bbox_inches='tight')
plt.show()

### Data-driven justification for optimal k=2

k=2 was selected as the optimal number of clusters based on the
following evidence:

1. **Silhouette score**: k=2 produced the highest mean silhouette
   score of 0.8151 across the entire range k=2 to k=40. A score
   of 0.8151 is strong, indicating the two clusters are well
   separated relative to their internal spread. No other value of k
   in the search range came close to this score.

2. **Elbow method**: the inertia curve drops steeply from k=1 to
   k=2 then flattens gradually. The sharpest reduction in inertia
   happens at k=2, after which adding more clusters gives
   diminishing returns.

3. **Agreement between methods**: both the elbow and silhouette
   methods independently point to k=2. When two different methods
   agree it gives stronger confidence in the selection than either
   method alone.

4. **Interpretability**: the two clusters map onto a real and
   meaningful distinction in handwriting style, confirming that
   k=2 reflects genuine structure in the data rather than an
   arbitrary split.

### Task 4 summary and cluster analysis

The K-means clustering of 900 digit 7 test samples reveals
meaningful structure within what looks like a single class.

Key observations:

- The two clusters separate standard digit 7s (886 samples, no
  crossbar) from European-style digit 7s (14 samples, with a
  horizontal crossbar through the diagonal stroke). This is a real
  and interpretable handwriting style difference, not a random split.
- The heavily uneven cluster sizes (886 vs 14) reflect the actual
  frequency of these two writing conventions in the dataset.
  The plain 7 dominates in English-language datasets, which is
  expected and meaningful rather than a flaw in the clustering.
- The silhouette score of 0.8151 confirms the two groups are
  genuinely distinct. This is not one cluster being arbitrarily
  forced into two halves.
- The elbow curve flattens gradually rather than showing a sharp
  bend, which is typical of handwriting data where variation is
  continuous rather than forming hard discrete groups.
- This shows that even within a single digit class there is enough
  visual variation for K-means to find interpretable subgroups.

---
## Summary and Reflection

This notebook covered binary classification and clustering analysis
on a subset of the MNIST dataset containing digits 1 and 7.

### What was done

**Task 1** confirmed the training set is balanced (50 samples per
class) while the test set is imbalanced (100 digit 1, 900 digit 7).
Pixel statistics showed digit 7 has a higher mean (29.53 vs 20.18)
and 48% more non-zero pixels (16.76% vs 11.27%), confirming strong
linear separability. PCA projection to 2 dimensions showed clean
class separation, directly motivating the linear model choice.

**Task 2** trained a Logistic Regression classifier. The choice was
justified by the small training set size and the visual separability
of the two digit classes. The model achieved strong accuracy and
macro F1 on the test set. The weight visualisation confirmed the
model correctly learned to focus on the horizontal bar region for
digit 7 and the narrow vertical region for digit 1.

**Task 3** compared Logistic Regression against a Kernel SVM with
RBF kernel. Both models were tuned using 5-fold stratified cross
validation on the training set without touching the test data.
The SVM required tuning of two parameters, C and gamma, via a grid
search. Both models performed comparably on this task given how
separable the classes are, which confirms the dataset does not
require a non-linear boundary.

**Task 4** applied K-means clustering to digit 7 test samples.
Starting with k=100 as required, the elbow method and silhouette
score were used to identify the optimal number of clusters. The
cluster centres and representative samples revealed real variation
in how people write digit 7, including differences in stroke width,
slant, and the presence or absence of a horizontal crossbar.

### Limitations and further work

- The 100-sample training set limits model complexity. Cross-
  validation estimates on such small data have high variance even
  with stratification, so reported CV scores should be interpreted
  with that in mind.
- Both classifiers used flat 784-dimensional pixel vectors. A small
  CNN would learn spatial neighbourhood features directly and likely
  outperform both linear and kernel approaches given more data.
- The clustering used raw standardised pixel features. Applying PCA
  dimensionality reduction before K-means would reduce noise and
  likely produce more stable and compact cluster assignments.
- Only elbow and silhouette methods were used for optimal k selection.
  The Davies-Bouldin index or gap statistic could be added to
  cross-validate the choice of k=2.
- The test set class imbalance (100 vs 900) means per-class accuracy
  for digit 1 has higher variance. A larger balanced test set would
  give more reliable per-class estimates.

In [ ]:
# final summary of all key results
print('=== Final Results Summary ===')
print()
print(f'Task 2  Logistic Regression (C=1.0)')
print(f'        Accuracy : {acc_lr:.4f}')
print(f'        Macro F1 : {f1_lr:.4f}')
print()
print(f'Task 3  Best LR   C={best_C_lr}')
print(f'        Accuracy : {acc_lr_f:.4f}')
print(f'        Macro F1 : {f1_lr_f:.4f}')
print()
print(f'Task 3  Best SVM  C={best_C_svm}  gamma={best_gamma_svm}')
print(f'        Accuracy : {acc_svm_f:.4f}')
print(f'        Macro F1 : {f1_svm_f:.4f}')
print()
print(f'Task 4  Digit 7 test samples : {X_test_7.shape[0]}')
print(f'        K-means k=100 inertia: {km100.inertia_:.2f}')
print(f'        Optimal k            : {optimal_k}')
print(f'        Silhouette score     : {optimal_sil}')